# Tarea Módulo VII

Entrena un agente usando Q-learning en un ambiente de gymnasium, (Classical control, Box2d, Toy Text) el cual no hayamos utilizado en clase

In [3]:
import gymnasium as gym
import numpy as np
from random import randint

In [61]:
env= gym.make("Blackjack-v1")

En la documentación de gymnasium para blackjack nos dice que nuestro entorno es discreto, el Observation Space es Tuple(Discrete(32), Discrete(11), Discrete(2)) y el Action space es Discrete(2).

Por lo tanto no tenemos ningúna variable continua por lo tanto no necesitamos discretizar. 

In [62]:
# Inicializamos nuestra Q-table con zeros
q_table = np.zeros((32, 11, 2, 2)) 

# Donde:
#    32 -> Valores posible de suma del jugador de 0 a 31, la idea es acercarse al 21. Se le da un rango de hasta 31 por si se pasa.
#    11 -> Cartas del dealer
#    2 -> Valores True/Flase para saber si usas el As como 11 o 1
#    2 -> Acciones posibles del jugador: Ya no tomar cartas (Stick) y Pedir una carta (Hit)

De acuerdo con la documentación las recompensas que el modelo puede tomar son:
* Ganar: +1
* Perder: -1
* Empatar: 0

In [63]:
alfa = 0.1
gamma = 0.95
episodios = 50000
epsilon=1
lista_recompenzas = []
for episodio in range(episodios):
    estado = env.reset()[0]
    final= False
    recompensa_total = 0
    
    while not final:
        player_sum, dealer_card, usable_ace = estado
        usable_ace = int(usable_ace)

        if randint(0,10)>epsilon:
            accion = np.argmax(q_table[player_sum][dealer_card][usable_ace])
        else:
            accion =randint(0,1) # Acciones posible Stick: 0 y Hit: 1

        nuevo_estado,recompensa,terminated,truncated,info = env.step(accion)
        final = terminated or truncated
        
        if not final:
            new_player_sum, new_dealer_card, new_usable_ace = nuevo_estado
            new_usable_ace = int(new_usable_ace)
            max_q = np.max(q_table[new_player_sum][new_dealer_card][new_usable_ace])
        else:
            max_q = 0

        q_table[player_sum][dealer_card][usable_ace][accion] = q_table[player_sum][dealer_card][usable_ace][accion] + alfa*(recompensa+gamma*max_q-q_table[player_sum][dealer_card][usable_ace][accion])
        estado= nuevo_estado
        recompensa_total+=recompensa

    lista_recompenzas.append(recompensa_total)
    if (episodio+1)%1000==0:
        print("Episodio: "+str(episodio)+"| Recompensa promedio: "+str(np.mean(lista_recompenzas)))

env.close()

Episodio: 999| Recompensa promedio: -0.241
Episodio: 1999| Recompensa promedio: -0.2205
Episodio: 2999| Recompensa promedio: -0.20766666666666667
Episodio: 3999| Recompensa promedio: -0.189
Episodio: 4999| Recompensa promedio: -0.178
Episodio: 5999| Recompensa promedio: -0.17283333333333334
Episodio: 6999| Recompensa promedio: -0.169
Episodio: 7999| Recompensa promedio: -0.1645
Episodio: 8999| Recompensa promedio: -0.16233333333333333
Episodio: 9999| Recompensa promedio: -0.1601
Episodio: 10999| Recompensa promedio: -0.15372727272727274
Episodio: 11999| Recompensa promedio: -0.1515
Episodio: 12999| Recompensa promedio: -0.152
Episodio: 13999| Recompensa promedio: -0.15157142857142858
Episodio: 14999| Recompensa promedio: -0.14786666666666667
Episodio: 15999| Recompensa promedio: -0.1435625
Episodio: 16999| Recompensa promedio: -0.14188235294117646
Episodio: 17999| Recompensa promedio: -0.1456111111111111
Episodio: 18999| Recompensa promedio: -0.1463684210526316
Episodio: 19999| Recompe

Ponemos a prueba el modelo jugando una partida e imrpimiremos en pantalla si el modelo gano, empato o perdió.

Para poder saber las acciones del modelo imprimiré en pantalla el estado actual y el estado después de que el modelo decidierá su acción. 

In [91]:
env= gym.make("Blackjack-v1", render_mode="human")
estado=env.reset()[0]
final= False
recompensa_total = 0

while not final:
    player_sum, dealer_card, usable_ace = estado
    usable_ace = int(usable_ace)
    accion = np.argmax(q_table[player_sum][dealer_card][usable_ace])

    nuevo_estado, recompensa, terminated, truncated, info = env.step(accion)
    
    print(f"------------------------")
    print(f"Estado actual: {estado}")
    print(f"Acción: {'Stick' if accion == 0 else 'Hit'}")
    print(f"Nuevo estado: {nuevo_estado}")
    print(f"Recompensa: {recompensa}")
    
    estado = nuevo_estado
    recompensa_total += recompensa
    final = terminated or truncated
env.close()

if recompensa_total == 1:
    print("El modelo gano la partida")
elif recompensa_total == 0:
    print("El modelo empato")
else:
    print("El modelo perdió")

------------------------
Estado actual: (16, 10, 1)
Acción: Hit
Nuevo estado: (16, 10, 0)
Recompensa: 0.0
------------------------
Estado actual: (16, 10, 0)
Acción: Hit
Nuevo estado: (20, 10, 0)
Recompensa: 0.0
------------------------
Estado actual: (20, 10, 0)
Acción: Stick
Nuevo estado: (20, 10, 0)
Recompensa: 1.0
El modelo gano la partida


Para poder visualizar la animación de juego se necesita más de una partida, por lo que haremos un ciclo de 50 partidas, para ir viendo los resultados de las decisiones del modelo. 

In [ ]:
env= gym.make("Blackjack-v1", render_mode="human")
victorias = 0
derrotas = 0
empates = 0

for _ in range(50):
    estado = env.reset()[0]
    final = False

    while not final:
        player_sum, dealer_card, usable_ace = estado
        usable_ace = int(usable_ace)

        accion = np.argmax(q_table[player_sum][dealer_card][usable_ace])
        estado, recompensa, terminated, truncated, _ = env.step(accion)
        final = terminated or truncated

    if recompensa == 1:
        victorias += 1
    elif recompensa == 0:
        empates += 1
    else:
        derrotas += 1
env.close()
print("Victorias:", victorias)
print("Empates:", empates)
print("Derrotas:", derrotas)

Victorias: 28
Empates: 5
Derrotas: 17


Los resultados de poner a jugar al modelo 50 partidas son bastantes buenos desde mi punto de vista, ya que al final el BlackJack es un juego que tiene matématica y probabilidad estadística por detrás pero sigue teniendo ese factor de suerte que lo vuelve un juego de azar al final del día. Por eso es que sería díficil conseguir un modelo que gane todas sus partidas.